## This notebook has PySpark code demonstrating how to work with complex data types such as:
1. Arrays
2. Structs
3. JSON data 

### The notebook deals with the following for the data-types
1. ArrayType Operations  (split_lenngth, contains, explode)
2. StructType with arrays
3. JSON operations (get_json_object, from_json)

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, IntegerType
from pyspark.sql import functions as F

# Sample data
rows = [
    Row(
        id=1,
        names=["Alice", "Bob"],
        scores=[85, 90],
        info=Row(age=25, city="New York"),
        json_str='{"dept": "Engineering", "level": 2}'
    ),
    Row(
        id=2,
        names=["Carol"],
        scores=[78, 88, 92],
        info=Row(age=30, city="San Francisco"),
        json_str='{"dept": "HR", "level": 1}'
    )
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("names", ArrayType(StringType()), True),
    StructField("scores", ArrayType(IntegerType()), True),
    StructField("info", StructType([
        StructField("age", IntegerType(), True),
        StructField("city", StringType(), True)
    ]), True),
    StructField("json_str", StringType(), True)
])

df = spark.createDataFrame(rows, schema)
df.createOrReplaceTempView("df")
display(df)


In [0]:
from pyspark.sql import functions as f

#get the length of names array
array_length_df=df.withColumn("names_length", f.size("names"))
#check if names array contains alice
array_contains_df=array_length_df.withColumn("name_contain_alice", f.array_contains("names", "Alice"))
#split a string column into an array 
df_array=array_contains_df.withColumn("city_words",f.split("info.city", " "))

df_array.select("id","names","names_length","name_contain_alice","city_words").display()

In [0]:
%sql
-- For performing the similar Spark SQL operations in PSQL we do these commands. Similar use but different commands. No such equivalence present for MySQL
-- SELECT
--     cardinality(names) AS names_length,
--     'Alice' = ANY(names) AS contains_alice,
--     string_to_array(city, ' ') AS city_words
-- FROM employees;
select 
    id,
    names,
    size(names) as names_length,
    array_contains(names, 'Alice') as name_contain_alice,
    split(info.city, ' ') as city_words
from df

In [0]:
from pyspark.sql import functions as f

exploded_df=df.select("id", f.explode("names").alias("name"))
exploded_df.display()

In [0]:
%sql
select 
    id,
    explode(names) as name
from df

In [0]:
#accessing a struct field and its value
from pyspark.sql import functions as f
struct_df=df.withColumn("info_age", f.col("info.age")).\
    withColumn("info_city",f.col("info.city"))
struct_df.display()

In [0]:
%sql
select *,
    info.age as info_age,
    info.city as info_city
from df

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Extract a value using get_json_object
json_df = df.withColumn("dept", F.get_json_object(F.col("json_str"), "$.dept")) \
    .withColumn("level", F.get_json_object(F.col("json_str"), "$.level"))

display(json_df)

In [0]:
%sql
select *,
    get_json_object(json_str, '$.dept') as dept,
    get_json_object(json_str, '$.level') as level
from df

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


# Parse JSON string into struct
json_schema = StructType([
    StructField("dept", StringType(), True),
    StructField("level", IntegerType(), True)
])
from_json_df = df.withColumn("json_struct", F.from_json(F.col("json_str"), json_schema))
# Accessing fields in the 'info' struct
struct_df = from_json_df.withColumn("dept", F.col("json_struct.dept"))

display(struct_df)

In [0]:
%sql
select *,
    from_json(json_str, 'STRUCT<dept:STRING,level:INT>') as json_struct,
    from_json(json_str, 'STRUCT<dept:STRING,level:INT>').dept as dept
from df